In [3]:
import pandas as pd
import numpy as np
from tabulate import tabulate # 사용되지 않지만 구조 유지를 위해 유지

# 💡 일반 학습 (Non-Transfer Learning) 방법: '테스트 정확도 (Epoch)'로 정렬
SORT_KEY_NON_TL = '테스트 정확도 (Epoch)'
# 💡 전이 학습 (Transfer Learning) 방법: '테스트 정확도 (Olf Rem. Epoch)'로 정렬
SORT_KEY_TL = '테스트 정확도 (Olf Rem. Epoch)' # 파일에 이 칼럼이 있다면 사용됨

# 1. Method 2 파일 목록 정의 (업로드된 실제 파일명 사용)
method2_file_list = [
    "테스트 상민_Cond3.csv",
    "테스트 승민_Cond3.csv",
    "테스트 준혁_Cond3.csv",
    "테스트 형주_Cond3.csv",
    "테스트 승욱_Cond3.csv"
]

# ⭐️ 수정된 함수: glob 대신 명시적인 file_list를 인수로 받습니다.
def average_specific_files(file_list, method_name, output_filename, sort_by_column):
    
    if not file_list:
        print(f"⚠️ 경고: {method_name}의 파일을 찾을 수 없습니다.")
        return None

    print(f"\n--- [평균 계산] {method_name} ({len(file_list)}개 파일) ---")
    
    # 여러 파일을 모두 불러와 하나의 DataFrame으로 합치기
    df_list = []
    for f in file_list:
        try:
            # 'None' 문자열을 NaN으로 읽지 않도록 설정 (기존 코드 유지)
            df = pd.read_csv(f, keep_default_na=False)
            df_list.append(df)
            print(f"   -> 파일 로드 완료: {f}")
        except Exception as e:
            print(f"❌ 파일 로드 중 오류 발생 ({f}): {e}")
            continue
            
    if not df_list:
        print(f"❌ 오류: 로드할 유효한 데이터가 없습니다.")
        return None
            
    combined_df = pd.concat(df_list, ignore_index=True)
    
    group_keys = ['특징', '밴드', '정규화', '모델']
    
    # 1. 평균 계산에 사용할 수 있는 모든 정확도 칼럼을 숫자로 변환
    accuracy_cols = ['학습 정확도 (Trial)', '학습 정확도 (Epoch)', 
                     '테스트 정확도 (Trial)', '테스트 정확도 (Epoch)']
    
    # 만약 전이 학습 관련 칼럼이 있다면 추가 (안전 장치)
    if '테스트 정확도 (Olf Rem. Epoch)' in combined_df.columns:
        accuracy_cols.append('테스트 정확도 (Olf Rem. Epoch)')
    
    # 숫자로 변환할 수 없는 값(예: 텍스트)은 NaN으로 처리
    for col in accuracy_cols:
        if col in combined_df.columns:
            combined_df[col] = pd.to_numeric(
                combined_df[col], 
                errors='coerce' # 숫자로 변환할 수 없는 값은 NaN으로 처리
            )
            
    # 2. 그룹별 평균 계산 (NaN 값은 자동으로 무시하고 유효한 값만으로 평균 계산)
    # numeric_only=True로 '혼동 행렬' 같은 문자열 칼럼은 제외됨
    average_results_df = combined_df.groupby(group_keys).mean(numeric_only=True)
    
    # 3. 정렬 및 인덱스 재설정
    final_sorted_df = average_results_df.reset_index()
    
    if sort_by_column in final_sorted_df.columns:
         final_sorted_df = final_sorted_df.sort_values(
             by=sort_by_column, ascending=False
         )
    else:
        print(f"❌ 오류: 정렬 키 '{sort_by_column}'이 최종 결과에 없습니다. 정렬 없이 저장합니다.")

    # 소수점 4자리까지 반올림
    numeric_cols = final_sorted_df.select_dtypes(include=np.number).columns
    final_sorted_df[numeric_cols] = final_sorted_df[numeric_cols].round(4)
    
    # 파일 저장
    final_sorted_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    print(f"✅ 평균 결과가 '{output_filename}'로 저장되었습니다.")
    
    # 상위 10개 결과 출력 (참고용)
    print("\n**🥇 상위 10개 결과 (참고용):**")
    print(tabulate(final_sorted_df.head(10), headers='keys', tablefmt='fancy_grid', showindex=False, numalign="right"))

    return final_sorted_df

# ==========================================================
# 🚀 5인 평균 계산 실행
# ==========================================================

### Method 2 평균 계산 (5개 파일만 존재하므로 이 부분만 실행합니다.)
avg_method2 = average_specific_files(
    file_list=method2_file_list,
    method_name="Method 2 (5인 평균)",
    output_filename="5인_Cond2.csv",
    sort_by_column=SORT_KEY_NON_TL
)


--- [평균 계산] Method 2 (5인 평균) (5개 파일) ---
   -> 파일 로드 완료: 상민_Cond2.csv
   -> 파일 로드 완료: 승민_Cond2.csv
   -> 파일 로드 완료: 준혁_Cond2.csv
   -> 파일 로드 완료: 형주_Cond2.csv
   -> 파일 로드 완료: 승욱_Cond2.csv
✅ 평균 결과가 '5인_Cond2.csv'로 저장되었습니다.

**🥇 상위 10개 결과 (참고용):**
╒═══════════════════╤════════╤══════════╤══════════════════════════╤═══════════════════════╤═══════════════════════╤═════════════════════════╤═════════════════════════╕
│ 특징              │ 밴드   │ 정규화   │ 모델                     │   학습 정확도 (Trial) │   학습 정확도 (Epoch) │   테스트 정확도 (Trial) │   테스트 정확도 (Epoch) │
╞═══════════════════╪════════╪══════════╪══════════════════════════╪═══════════════════════╪═══════════════════════╪═════════════════════════╪═════════════════════════╡
│ F_median_3d_8rm   │ alpha  │ norm3    │ Logistic Regression      │                0.6533 │                 0.621 │                    0.62 │                  0.5971 │
├───────────────────┼────────┼──────────┼──────────────────────────┼───────────────────────┼──────────────────

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings
import io
import os

# -----------------------------------------------------------------
# 0. 한글 폰트 설정 (Windows, Mac, Linux 자동 감지)
# -----------------------------------------------------------------
def set_korean_font():
    """운영체제에 따라 한글 폰트를 설정합니다."""
    # 폰트 우선순위: NanumGothic > AppleGothic > Malgun Gothic
    font_priorities = ['NanumGothic', 'AppleGothic', 'Malgun Gothic']
    
    font_found = False
    
    # Nanum 폰트 (리눅스 및 보편적 환경)
    if 'NanumGothic' in fm.findfont(fm.FontProperties(family='NanumGothic', weight='bold')):
        plt.rcParams['font.family'] = 'NanumGothic'
        font_found = True
        print(f"✅ 한글 폰트(NanumGothic)가 설정되었습니다.")

    # AppleGothic 폰트 (macOS)
    if not font_found and 'AppleGothic' in fm.findfont(fm.FontProperties(family='AppleGothic', weight='bold')):
        plt.rcParams['font.family'] = 'AppleGothic'
        font_found = True
        print(f"✅ 한글 폰트(AppleGothic)가 설정되었습니다.")

    # Malgun Gothic 폰트 (Windows)
    if not font_found and os.path.exists('c:/Windows/Fonts/malgun.ttf'):
        try:
            font_prop = fm.FontProperties(fname='c:/Windows/Fonts/malgun.ttf')
            plt.rcParams['font.family'] = font_prop.get_name()
            font_found = True
            print(f"✅ 한글 폰트(Malgun Gothic)가 설정되었습니다.")
        except Exception:
            pass # Malgun Gothic 로드 실패 시 무시

    if not font_found:
        print("⚠️ 한글 폰트를 찾을 수 없습니다. 그래프의 한글이 깨질 수 있습니다.")
        
    plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지
    warnings.filterwarnings('ignore') # 불필요한 경고 메시지 숨기기

set_korean_font()

# -----------------------------------------------------------------
# 1. 데이터 로드
# -----------------------------------------------------------------
# ⭐️⭐️⭐️ 분석할 5명 평균 Method 2 CSV 파일 이름을 여기에 입력하세요 ⭐️⭐️⭐️
file_name = "5인_Cond2.csv" 

try:
    # 'None' 문자열이 NaN(결측치)으로 변환되는 것을 방지합니다.
    df = pd.read_csv(file_name, keep_default_na=False) 
    
    # 정확도 컬럼을 숫자로 변환 (변환 불가 값은 NaN 처리)
    accuracy_cols = ['학습 정확도 (Trial)', '학습 정확도 (Epoch)', 
                     '테스트 정확도 (Trial)', '테스트 정확도 (Epoch)']
    for col in accuracy_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    print(f"\n'{file_name}' 파일을 성공적으로 로드했습니다.")
    print(f"원본 데이터 크기: {len(df)} 행")

except FileNotFoundError:
    print(f"❌ 오류: '{file_name}'을(를) 찾을 수 없습니다.")
    print("    파일 이름이 정확한지, 노트북과 같은 폴더에 있는지 확인하세요.")
    raise
except Exception as e:
    print(f"❌ 파일 로드 중 오류 발생: {e}")
    raise

# -----------------------------------------------------------------
# 2. 특징 및 채널/영역 추출 (데이터 전처리)
# -----------------------------------------------------------------
print("\n--- 데이터 전처리 중 (19채널 분류 기준 추가됨) ---")

# 2-1. 분류 기준 정의
feature_types = ['mean', 'median', 'peak', 'de', 'dasm', 'rasm']
# 12개 개별 채널
channel_types = ['Fz', 'F3', 'F4', 'F7', 'F8', 'Cz', 'C3', 'C4', 'T7', 'T8', 'Fp1', 'Fp2']
# 5개 그룹 특징
area_types = ['10rm', '8rm', '2area', '1area', '3area'] 

# 2-2. '특징_분류' (6가지) 추출 (mean, median, peak, de, dasm, rasm)
feat_pattern = f"({'|'.join(feature_types)})"
df['특징_분류'] = df['특징'].str.extract(feat_pattern, expand=False)

# 2-3. '채널_영역' (12개 채널 + 5개 영역 + 19ch_Group + 12ch_Group) 추출
# 1. 개별 채널 또는 명시된 영역 (총 17가지) 추출
chan_area_pattern = f"_({'|'.join(channel_types + area_types)})$"
df['채널_영역'] = df['특징'].str.extract(chan_area_pattern, expand=False)

# 2. '3d19' 패턴을 포함하며, 아직 채널/영역이 분류되지 않은 특징 ('19ch_Group') 분류
# 특징명에 '3d19'가 포함되어 있고, 채널/영역 컬럼이 아직 NaN인 경우 (예: F_rasm_3d19)
is_3d19_feature = df['특징'].str.contains('3d19', na=False)
is_nan_area = df['채널_영역'].isna()
df.loc[is_3d19_feature & is_nan_area, '채널_영역'] = '19ch_Group'

# 3. 나머지 채널/영역이 없는 경우 ('12ch_Group') 분류 (기존의 12ch_Group 추정 값)
# 12채널 전체를 사용하는 그룹 특징 등으로 추정
df['채널_영역'] = df['채널_영역'].fillna('12ch_Group') 

# 2-4. 추출 확인
print(f"추출된 특징 (6종): {df['특징_분류'].unique()}")
print(f"추출된 채널/영역 (총 {len(df['채널_영역'].unique())}종): {df['채널_영역'].unique()}")
print(f"분류된 채널/영역에 '19ch_Group'이 포함되었습니다.")


# -----------------------------------------------------------------
# 3. 데이터 필터링 (⭐️ 요청 조건으로 수정됨)
# -----------------------------------------------------------------
# ⭐️ 사용자가 요청한 컬럼 및 조건: 테스트 정확도 (Trial) >= 0.6
test_col_epoch = '테스트 정확도 (Trial)'

try:
    # 필터링 조건: 테스트 정확도 (Epoch)가 0.5 이상인 행만 선택
    filter_condition = (df[test_col_epoch] >= 0.6)
    df_filtered = df[filter_condition].copy()
    
    # 필터링 전, 정확도 컬럼들이 유효한 숫자인지 확인
    if df_filtered[test_col_epoch].isnull().any():
          print("⚠️ 경고: '테스트 정확도 (Epoch)' 컬럼에 NaN 값이 포함되어 있습니다. 이는 필터링 시 자동으로 제외됩니다.")
    
    print(f"\n--- 필터링 결과 ---")
    print(f"총 {len(df)}개 조합 중 {len(df_filtered)}개 조합이 필터 조건을 만족했습니다.")
    print(f"(조건: {test_col_epoch} >= 0.5)")

except KeyError as e:
    print(f"❌ 필터링 오류: '{e}' 컬럼을 찾을 수 없습니다.")
    print("    컬럼명이 정확한지 확인하세요. (Method 2 컬럼명)")
    df_filtered = pd.DataFrame() # 빈 DataFrame으로 설정하여 그래프 생성을 스킵


# -----------------------------------------------------------------
# 4. 공통 그래프 생성 함수 (⭐️ 학습 막대 제거, 테스트 정확도 (Epoch) 사용)
# -----------------------------------------------------------------
def create_single_accuracy_plot_with_counts(data, group_by_col, x_label, title, filename):
    """
    지정된 컬럼으로 그룹화하여 평균 '테스트 정확도 (Trial)'만 막대그래프로 표기합니다.
    (Y축 간격 0.05, 카운트 폰트 크기 증가)
    """
    if data.empty:
        print(f"\n⚠️ 경고: '{title}' 그래프는 필터링된 데이터가 없어 생성할 수 없습니다.")
        return

    # 1. 그룹별 평균 계산 (Mean) - 테스트 정확도 (Epoch)만 사용
    avg_data = data.groupby(group_by_col)[[test_col_epoch]].mean().reset_index()
    
    # 2. 그룹별 개수 계산 (Count)
    count_data = data.groupby(group_by_col).size().reset_index(name='count')
    
    # 3. 평균(avg_data)과 개수(count_data)를 병합
    plot_data = pd.merge(avg_data, count_data, on=group_by_col)
    
    # 4. 데이터를 '테스트 정확도 (Epoch)' 평균값을 기준으로 정렬
    plot_data = plot_data.sort_values(by=test_col_epoch, ascending=False).reset_index(drop=True)

    # 5. 막대그래프 생성 (ax 객체에 저장) - 파란색 막대만 사용
    plt.figure(figsize=(14, 8))
    ax = sns.barplot(
        data=plot_data,
        x=group_by_col,
        y=test_col_epoch,
        color='blue', # 파란색으로 고정
        order=plot_data[group_by_col] # 이미 정렬된 순서 사용
    )
    
    # 6. 그래프 설정
    plt.title(f"{title}\n(필터 적용: {test_col_epoch} >= 0.5)", fontsize=16)
    plt.xlabel(x_label, fontsize=12)
    plt.ylabel("평균 테스트 정확도 (Trial)", fontsize=12) # Y축 레이블 변경
    
    # Y축 간격 0.05로 수정
    plt.ylim(0.45, plot_data[test_col_epoch].max() + 0.05) # Y축 최소값을 0.45로 설정하여 0.5 필터링 효과 시각화
    plt.yticks(np.arange(0, 1.05, 0.05), fontsize=11) 
    
    plt.xticks(rotation=45, ha='right', fontsize=11)
    # plt.legend() 제거 (학습 정확도 막대 제거로 범례 불필요)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # 7. 막대 위에 카운트(개수) 표기 (폰트 크기 14)
    for i, bar in enumerate(ax.patches):
        height = bar.get_height() # 막대의 높이
        count = plot_data.loc[i, 'count'] # 해당 그룹의 개수
        ax.annotate(
            f'{count}',                          # 표기할 텍스트 (조합의 개수)
            (bar.get_x() + bar.get_width() / 2, height), # (x, y) 위치
            ha='center',                         # 수평 정렬
            va='bottom',                         # 수직 정렬
            fontsize=14,                         # 폰트 크기를 14로 키움
            fontweight='bold',                   # 굵게 표시
            color='black',
            xytext=(0, 2), # 막대 상단에서 2포인트 위에 표기
            textcoords='offset points'
        )
    
    plt.tight_layout() 
    
    # 8. 파일 저장 (파일 이름 변경: _TestEpoch_05_Only.png 추가)
    output_filename = filename.replace('.png', '_TestTrial_06_Only.png')
    plt.savefig(output_filename)
    print(f"✅ '{output_filename}' 그래프가 저장되었습니다.")
    plt.close() 

# -----------------------------------------------------------------
# 5. 5가지 그래프 생성 실행
# -----------------------------------------------------------------
if not df_filtered.empty:
    print("\n--- 5개 그래프 생성 시작 (테스트 정확도(Trial) >= 0.6 필터 적용) ---")
    
    # 1) x축: 채널/영역 (총 19가지)
    create_single_accuracy_plot_with_counts(
        data=df_filtered,
        group_by_col='채널_영역',
        x_label='채널 / 영역 (총 19가지 - 12개 채널 + 5개 영역 + 19ch Group + 12ch Group)',
        title='[Method 2] 채널/영역 별 평균 테스트 정확도 (Epoch) (0.5 이상 조합) (19채널 그룹 포함)',
        filename='Cond2_channel_area.png'
    )

    # 2) x축: 특징 (6가지)
    create_single_accuracy_plot_with_counts(
        data=df_filtered,
        group_by_col='특징_분류',
        x_label='특징 유형 (6가지)',
        title='[Method 2] 특징 유형 별 평균 테스트 정확도 (Trial) (0.6 이상 조합)',
        filename= 'Cond2_feature.png'
    )

    # 3) x축: 모델 (6가지)
    create_single_accuracy_plot_with_counts(
        data=df_filtered,
        group_by_col='모델',
        x_label='모델 (6가지)',
        title='[Method 2] 모델 별 평균 테스트 정확도 (Trial) (0.6 이상 조합)',
        filename='Cond2_model.png'
    )

    # 4) x축: 정규화 (3가지)
    create_single_accuracy_plot_with_counts(
        data=df_filtered,
        group_by_col='정규화',
        x_label='정규화 (3가지)',
        title='[Method 2] 정규화 방식 별 평균 테스트 정확도 (Trial) (0.6 이상 조합)',
        filename='Cond2_normalization.png'
    )

    # 5) x축: 밴드 (6가지)
    create_single_accuracy_plot_with_counts(
        data=df_filtered,
        group_by_col='밴드',
        x_label='주파수 밴드 (6가지)',
        title='[Method 2] 밴드 별 평균 테스트 정확도 (Trial) (0.6 이상 조합)',
        filename='Cond2_band.png'
    )
else:
    print("\n--- 그래프 생성 스킵 ---")
    print("⚠️ 필터 조건을 만족하는 데이터가 0개이므로 그래프를 생성하지 않습니다.")

print("\n--- Method 2 분석 작업 완료 ---")


✅ 한글 폰트(Malgun Gothic)가 설정되었습니다.

'5인_Cond2.csv' 파일을 성공적으로 로드했습니다.
원본 데이터 크기: 9288 행

--- 데이터 전처리 중 (19채널 분류 기준 추가됨) ---
추출된 특징 (6종): ['median' 'mean' 'dasm' 'de' 'peak' 'rasm']
추출된 채널/영역 (총 19종): ['8rm' '19ch_Group' 'F3' '3area' '12ch_Group' '10rm' '2area' 'T8' 'Fp2'
 'F7' 'F8' '1area' 'Fp1' 'Fz' 'Cz' 'C3' 'T7' 'F4' 'C4']
분류된 채널/영역에 '19ch_Group'이 포함되었습니다.

--- 필터링 결과 ---
총 9288개 조합 중 97개 조합이 필터 조건을 만족했습니다.
(조건: 테스트 정확도 (Trial) >= 0.5)

--- 5개 그래프 생성 시작 (테스트 정확도(Trial) >= 0.6 필터 적용) ---
✅ 'Cond2_channel_area_TestTrial_06_Only.png' 그래프가 저장되었습니다.
✅ 'Cond2_feature_TestTrial_06_Only.png' 그래프가 저장되었습니다.
✅ 'Cond2_model_TestTrial_06_Only.png' 그래프가 저장되었습니다.
✅ 'Cond2_normalization_TestTrial_06_Only.png' 그래프가 저장되었습니다.
✅ 'Cond2_band_TestTrial_06_Only.png' 그래프가 저장되었습니다.

--- Method 2 분석 작업 완료 ---
